In [2]:
import os
import numpy as np
import tifffile as tiff
#from tqdm.notebook import tqdm
from tqdm import tqdm

from cellpose import models

# --- paths ---
input_dir = "/mnt/e/segmentation_training/model_eval_cellseg_v2/test_images/tiffs/005_clahe/005_clahe/"
output_dir = "/mnt/e/segmentation_training/model_eval_cellseg_v2/test_images/tiffs/005_clahe/005_clahe/"
os.makedirs(output_dir, exist_ok=True)

# --- load model ---
model = models.CellposeModel(
    gpu=True,
    pretrained_model = "/mnt/e/segmentation_training/model_eval_cellseg_v2/model/cell_seg_v2_fullimg_epoch_0170"
)

# --- optimal parameters ---
stitch_threshold = 0.4
cellprob_threshold = 0.0
flow_threshold = 0.2

# --- get files ---
tiff_files = sorted([
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".tif", ".tiff"))
])
print(f"Found {len(tiff_files)} TIFF files.")

# --- run inference ---
for filename in tqdm(tiff_files):
    input_path = os.path.join(input_dir, filename)
    output_path = os.path.join(output_dir, filename.replace(".tif", "_masks.tif"))

    print(f"\nProcessing {filename}")
    volume = tiff.imread(input_path)
    print("Shape:", volume.shape)

    masks, flows, styles = model.eval(
        volume,
        diameter=None,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        do_3D=False,                      # 2D per-slice, matching training
        stitch_threshold=stitch_threshold, # links masks across z
        channels=[0, 0],
        z_axis=0,
    )

    tiff.imwrite(output_path, masks.astype(np.uint16), compression="zlib")
    print("Saved:", output_path)

print("\n✓ All done!")

Found 1 TIFF files.


  0%|          | 0/1 [00:00<?, ?it/s]


Processing drmGAL4_palmmKate_R5_005_40X_clahe_dn.tif


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Shape: (130, 1024, 1024)


100%|██████████| 1/1 [03:47<00:00, 227.12s/it]

Saved: /mnt/e/segmentation_training/model_eval_cellseg_v2/test_images/tiffs/005_clahe/005_clahe/drmGAL4_palmmKate_R5_005_40X_clahe_dn_masks.tif

✓ All done!
